<a href="https://colab.research.google.com/github/GopalKrishna-India/Geospatial/blob/master/FarmerFieldData-RFAZ/FarmerFieldCleaningRFAZ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

# 1. Load your KoboToolbox Excel export
input_excel = "Farmer_Field__Geoshape_Data_Collection2026-11-23.xlsx"
boundary_col = "field_boundary"
remarks_col = "remarks"

df = pd.read_excel(input_excel)

# 2. Function to standardize 'remarks' using your exact filter keywords
def standardize_remark(val):
    if pd.isna(val) or not str(val).strip():
        return None

    # Clean whitespace and trailing punctuation (e.g., "Alley plantation,")
    text = str(val).lower().strip().rstrip(",.")

    # Alley Plantation matches
    if "alley" in text:
        return "Alley Plantation"

    # Bund Plantation matches (includes "Bond", "Bond plant", "Bond plantation")
    if "bund" in text or "bond" in text:
        return "Bund Plantation"

    # Nutri Gardens matches (includes "N,G", "NG", "Nutri Garden", "Nutri garden plantation")
    if "nutri" in text or text in ["ng", "n,g", "n.g"]:
        return "Nutri Gardens"

    # Unrelated remarks ("Dh", "Farm Pond", "Good health of trees", "Shdh", etc.)
    return None

# Preserve original remarks for reference and apply standardization
df["remarks_original"] = df[remarks_col]
df[remarks_col] = df[remarks_col].apply(standardize_remark)

# Separate valid rows from discarded rows so discarded ones go to the bottom
valid_remarks_df = df[df[remarks_col].notna()].copy()
discarded_remarks_df = df[df[remarks_col].isna()].copy()
discarded_remarks_df[remarks_col] = "DISCARDED"

# Recombine with valid records on top and discarded records at the bottom
df = pd.concat([valid_remarks_df, discarded_remarks_df], ignore_index=True)

# 3. Function to separate Elevation and 2D Lat/Lon Polygon Geometry
def parse_kobo_geoshape(coord_str):
    if pd.isna(coord_str) or not str(coord_str).strip():
        return None, None, None, None, None

    lat_lon_clean = []
    poly_points = []
    elevations = []

    vertices = [v.strip() for v in str(coord_str).strip().split(";") if v.strip()]

    for vertex in vertices:
        parts = vertex.split()
        if len(parts) >= 2:
            lat = float(parts[0])
            lon = float(parts[1])

            if len(parts) >= 3:
                elevations.append(round(float(parts[2]), 2))

            lat_lon_clean.append(f"{lat} {lon}")
            # GeoJSON requires (Longitude, Latitude)
            poly_points.append((lon, lat))

    if len(poly_points) < 3:
        return None, None, None, None, None

    # Close polygon ring
    if poly_points[0] != poly_points[-1]:
        poly_points.append(poly_points[0])

    geom = Polygon(poly_points)
    if not geom.is_valid:
        geom = geom.buffer(0)

    clean_boundary_str = ";".join(lat_lon_clean)
    elev_list_str = ";".join(map(str, elevations)) if elevations else None
    avg_elev = round(sum(elevations) / len(elevations), 2) if elevations else None
    elev_range = f"{min(elevations)} - {max(elevations)}" if elevations else None

    return geom, clean_boundary_str, elev_list_str, avg_elev, elev_range

# Apply boundary & elevation separation
parsed = df[boundary_col].apply(parse_kobo_geoshape)

df["geometry"] = [p[0] for p in parsed]
df["lat_lon_boundary"] = [p[1] for p in parsed]
df["node_elevations_m"] = [p[2] for p in parsed]
df["avg_elevation_m"] = [p[3] for p in parsed]
df["elevation_range_m"] = [p[4] for p in parsed]

# Drop the original combined boundary column
df = df.drop(columns=[boundary_col])

# 4. Save updated Excel file (valid rows at top, DISCARDED rows at bottom)
df.drop(columns=["geometry"]).to_excel("farmer_fields_final.xlsx", index=False)

# 5. Export valid polygons to GeoJSON for GeoLibre Desktop
# Note: Excludes 'DISCARDED' rows from the map. Remove the second condition if you want them mapped too.
geo_df = df[(df["geometry"].notna()) & (df[remarks_col] != "DISCARDED")].copy()

gdf = gpd.GeoDataFrame(geo_df, geometry="geometry", crs="EPSG:4326")
gdf.to_file("farmer_fields_final.geojson", driver="GeoJSON")

print("Summary of 'remarks' in Excel:")
print(df[remarks_col].value_counts())
print(f"\nExported {len(gdf)} valid field polygons to farmer_fields_final.geojson")

Summary of 'remarks' in Excel:
remarks
Alley Plantation    164
Bund Plantation     103
Nutri Gardens        61
DISCARDED             5
Name: count, dtype: int64

Exported 328 valid field polygons to farmer_fields_final.geojson


Existing Python script (or run it directly on your cleaned gdf GeoDataFrame) to loop through the three standardized categories and export each one as its own separate, valid FeatureCollection GeoJSON file.

In [4]:
import geopandas as gpd

# Define your three standardized categories and clean output filenames
categories = {
    "Alley Plantation": "alley_plantation.geojson",
    "Bund Plantation": "bund_plantation.geojson",
    "Nutri Gardens": "nutri_gardens.geojson"
}

# Loop through each category, filter valid geometries, and export separately
for category_name, output_filename in categories.items():
    subset_gdf = gdf[(gdf["remarks"] == category_name) & (gdf["geometry"].notna())].copy()

    if not subset_gdf.empty:
        subset_gdf.to_file(output_filename, driver="GeoJSON")
        print(f"Exported {len(subset_gdf)} polygons -> {output_filename}")
    else:
        print(f"No valid polygons found for '{category_name}'")

Exported 164 polygons -> alley_plantation.geojson
Exported 103 polygons -> bund_plantation.geojson
Exported 61 polygons -> nutri_gardens.geojson


Standalone Option (If Running Directly from Your Cleaned Excel File)
If you already saved farmer_fields_final.xlsx and just want a quick standalone script to read that Excel file and generate the three separate GeoJSON files without re-running the entire pipeline:

In [ ]:

import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

# 1. Load your already-cleaned Excel file
df = pd.read_excel("farmer_fields_final.xlsx")

# 2. Rebuild polygons from the cleaned 'lat_lon_boundary' column
def rebuild_polygon(coord_str):
    if pd.isna(coord_str) or not str(coord_str).strip():
        return None
    points = []
    for vertex in str(coord_str).strip().split(";"):
        parts = vertex.strip().split()
        if len(parts) >= 2:
            lat, lon = float(parts[0]), float(parts[1])
            points.append((lon, lat))  # GeoJSON requires (Lon, Lat)
    if len(points) < 3:
        return None
    if points[0] != points[-1]:
        points.append(points[0])
    geom = Polygon(points)
    return geom if geom.is_valid else geom.buffer(0)

df["geometry"] = df["lat_lon_boundary"].apply(rebuild_polygon)
gdf = gpd.GeoDataFrame(df.dropna(subset=["geometry"]), geometry="geometry", crs="EPSG:4326")

# 3. Export each category as a separate GeoJSON file
categories = {
    "Alley Plantation": "alley_plantation.geojson",
    "Bund Plantation": "bund_plantation.geojson",
    "Nutri Gardens": "nutri_gardens.geojson"
}

for category_name, output_filename in categories.items():
    subset_gdf = gdf[gdf["remarks"] == category_name].copy()
    if not subset_gdf.empty:
        subset_gdf.to_file(output_filename, driver="GeoJSON")
        print(f"Exported {len(subset_gdf)} polygons -> {output_filename}")